In [1]:
# --- cell 1: setup ---
import sys, os, time, requests, pandas as pd
sys.path.append("..")            # so we can import config.py from notebooks/
from config import TOKEN

BASE   = "https://api.football-data.org/v4"
H      = {"X-Auth-Token": TOKEN}
SEASON = 2023
LEAGUES = {"PL":"Premier League", "PD":"La Liga", "BL1":"Bundesliga",
           "SA":"Serie A", "FL1":"Ligue 1"}

def get(path, params=None):
    r = requests.get(f"{BASE}{path}", headers=H, params=params)
    r.raise_for_status()
    time.sleep(7)                # stay under 10 calls/min
    return r.json()

In [2]:
# --- cell 2: standings (all leagues) ---
frames = []
for code, name in LEAGUES.items():
    tbl = get(f"/competitions/{code}/standings", {"season": SEASON})["standings"][0]["table"]
    df = pd.DataFrame([{
        "league": name, "position": t["position"], "team": t["team"]["name"],
        "played": t["playedGames"], "won": t["won"], "draw": t["draw"], "lost": t["lost"],
        "gf": t["goalsFor"], "ga": t["goalsAgainst"], "gd": t["goalDifference"],
        "points": t["points"],
    } for t in tbl])
    frames.append(df)
standings = pd.concat(frames, ignore_index=True)
standings.to_csv("../data/standings.csv", index=False)
standings.head()

,league,position,team,played,won,draw,lost,gf,ga,gd,points
0,Premier League,1,Manchester City FC,38,28,7,3,96,34,62,91
1,Premier League,2,Arsenal FC,38,28,5,5,91,29,62,89
2,Premier League,3,Liverpool FC,38,24,10,4,86,41,45,82
3,Premier League,4,Aston Villa FC,38,20,8,10,76,61,15,68
4,Premier League,5,Tottenham Hotspur FC,38,20,6,12,74,61,13,66


In [3]:
# --- cell 3: matches (all leagues) ---
frames = []
for code, name in LEAGUES.items():
    ms = get(f"/competitions/{code}/matches", {"season": SEASON})["matches"]
    df = pd.DataFrame([{
        "league": name, "matchday": m["matchday"], "date": m["utcDate"][:10],
        "home": m["homeTeam"]["name"], "away": m["awayTeam"]["name"],
        "home_goals": m["score"]["fullTime"]["home"],
        "away_goals": m["score"]["fullTime"]["away"], "status": m["status"],
    } for m in ms])
    frames.append(df)
matches = pd.concat(frames, ignore_index=True)
matches = matches[matches.status == "FINISHED"].copy()
matches["total_goals"] = matches.home_goals + matches.away_goals
matches["result"] = matches.apply(
    lambda r: "Home win" if r.home_goals > r.away_goals
    else ("Away win" if r.home_goals < r.away_goals else "Draw"), axis=1)
matches.to_csv("../data/matches.csv", index=False)
matches.head()

,league,matchday,date,home,away,home_goals,away_goals,status,total_goals,result
0,Premier League,1,2023-08-11,Burnley FC,Manchester City FC,0,3,FINISHED,3,Away win
1,Premier League,1,2023-08-12,Arsenal FC,Nottingham Forest FC,2,1,FINISHED,3,Home win
2,Premier League,1,2023-08-12,AFC Bournemouth,West Ham United FC,1,1,FINISHED,2,Draw
3,Premier League,1,2023-08-12,Brighton & Hove Albion FC,Luton Town FC,4,1,FINISHED,5,Home win
4,Premier League,1,2023-08-12,Everton FC,Fulham FC,0,1,FINISHED,1,Away win


In [4]:
# --- cell 4: scorers (all leagues) ---
frames = []
for code, name in LEAGUES.items():
    sc = get(f"/competitions/{code}/scorers", {"season": SEASON, "limit": 20})["scorers"]
    df = pd.DataFrame([{
        "league": name, "player": s["player"]["name"],
        "team": s["team"]["name"], "goals": s["goals"],
    } for s in sc])
    frames.append(df)
scorers = pd.concat(frames, ignore_index=True)
scorers.to_csv("../data/scorers.csv", index=False)
scorers.head()

,league,player,team,goals
0,Premier League,Erling Haaland,Manchester City FC,27
1,Premier League,Cole Palmer,Chelsea FC,22
2,Premier League,Alexander Isak,Newcastle United FC,21
3,Premier League,Dominic Solanke,AFC Bournemouth,19
4,Premier League,Phil Foden,Manchester City FC,19


In [5]:
# --- cell 5: reshape ---
home = matches.rename(columns={"home":"team","away":"opponent",
        "home_goals":"goals_for","away_goals":"goals_against"})
home["venue"] = "Home"
away = matches.rename(columns={"away":"team","home":"opponent",
        "away_goals":"goals_for","home_goals":"goals_against"})
away["venue"] = "Away"

tm = pd.concat([home, away], ignore_index=True)
tm["gd"]     = tm.goals_for - tm.goals_against
tm["points"] = tm.gd.apply(lambda x: 3 if x > 0 else (1 if x == 0 else 0))
tm["win"]    = (tm.gd > 0).astype(int)
tm = tm[["league","matchday","date","team","opponent","venue",
         "goals_for","goals_against","gd","points","win"]]
tm.to_csv("../data/team_matches.csv", index=False)
tm.head()

,league,matchday,date,team,opponent,venue,goals_for,goals_against,gd,points,win
0,Premier League,1,2023-08-11,Burnley FC,Manchester City FC,Home,0,3,-3,0,0
1,Premier League,1,2023-08-12,Arsenal FC,Nottingham Forest FC,Home,2,1,1,3,1
2,Premier League,1,2023-08-12,AFC Bournemouth,West Ham United FC,Home,1,1,0,1,0
3,Premier League,1,2023-08-12,Brighton & Hove Albion FC,Luton Town FC,Home,4,1,3,3,1
4,Premier League,1,2023-08-12,Everton FC,Fulham FC,Home,0,1,-1,0,0
